In [1]:
import pandas as pd

In [2]:
file_path = "raw/by-primary-language-certified-eligibles.csv"

df_raw = pd.read_csv(
    file_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

df_raw.shape

(251535, 9)

In [3]:
print(df_raw.columns.tolist())
print()
print(df_raw.dtypes)

['Month of Eligibility', 'County', 'Language', 'Total Eligibles', 'Annotation Code', 'Annotation Description', 'Status', 'Date of Data', 'Difference']

Month of Eligibility      str
County                    str
Language                  str
Total Eligibles           str
Annotation Code           str
Annotation Description    str
Status                    str
Date of Data              str
Difference                str
dtype: object


In [4]:
df_raw.head(10)

,Month of Eligibility,County,Language,Total Eligibles,Annotation Code,Annotation Description,Status,Date of Data,Difference
0,2026-08,Alameda,ASL,132,,,P,11AUG2026,
1,2026-08,Alameda,Arabic,"2,724",,,P,11AUG2026,
2,2026-08,Alameda,Armenian,35,,,P,11AUG2026,
3,2026-08,Alameda,Cambodian,715,,,P,11AUG2026,
4,2026-08,Alameda,Cantonese,"23,765",,,P,11AUG2026,
5,2026-08,Alameda,English,"287,771",,,P,11AUG2026,
6,2026-08,Alameda,Farsi,"3,101",,,P,11AUG2026,
7,2026-08,Alameda,French,179,,,P,11AUG2026,
8,2026-08,Alameda,Hebrew,,1,Cell suppressed for small numbers,P,11AUG2026,
9,2026-08,Alameda,Hindi,846,,,P,11AUG2026,


## Item 4 - Inspect Columns and Types

The raw CSV is intentionally loaded as text first so that source values are preserved before conversion.

| Column | Raw type | Intended type | Planned handling |
|---|---|---|---|
| Month of Eligibility | text | monthly period/date | Parse `YYYY-MM` strictly |
| County | text | categorical text | Preserve source labels |
| Language | text | categorical text | Preserve labels; review whitespace separately |
| Total Eligibles | text | nullable integer | Remove valid thousands separators and preserve suppressed values as missing |
| Annotation Code | text | categorical text | Keep codes `1` and `2` as labels, not numbers |
| Annotation Description | text | text | Preserve descriptions |
| Status | text | categorical text | Preserve `F` and `P` |
| Date of Data | text | nullable date | Parse valid `DDMONYYYY` values |
| Difference | text | unresolved / nullable | Entirely empty; do not infer a type or meaning |

In [5]:
month_parsed = pd.to_datetime(
    df_raw["Month of Eligibility"],
    format="%Y-%m",
    errors="coerce"
)

month_failed = month_parsed.isna().sum()

print("Failed month conversions:", month_failed)
print("Distinct parsed months:", month_parsed.nunique())
print("Min month:", month_parsed.min())
print("Max month:", month_parsed.max())

Failed month conversions: 0
Distinct parsed months: 200
Min month: 2010-01-01 00:00:00
Max month: 2026-08-01 00:00:00


In [6]:
total_raw = df_raw["Total Eligibles"]

total_clean_text = total_raw.str.strip().str.replace(",", "", regex=False)

total_numeric = pd.to_numeric(
    total_clean_text.replace("", pd.NA),
    errors="coerce"
)

print("Nonblank source values:", (total_clean_text != "").sum())
print("Successful numeric conversions:", total_numeric.notna().sum())
print("Missing / suppressed after conversion:", total_numeric.isna().sum())
print("Min numeric value:", total_numeric.min())
print("Max numeric value:", total_numeric.max())

Nonblank source values: 157211
Successful numeric conversions: 157211
Missing / suppressed after conversion: 94324
Min numeric value: 11.0
Max numeric value: 16051565.0


In [7]:
date_raw = df_raw["Date of Data"].str.strip()

date_clean = date_raw.replace(
    {
        "": pd.NA,
        ".": pd.NA
    }
)

date_parsed = pd.to_datetime(
    date_clean,
    format="%d%b%Y",
    errors="coerce"
)

nonmissing_date_source = date_clean.notna().sum()
successful_date_conversions = date_parsed.notna().sum()
failed_date_conversions = (
    date_clean.notna() & date_parsed.isna()
).sum()

print("Nonmissing date source values:", nonmissing_date_source)
print("Successful date conversions:", successful_date_conversions)
print("Failed date conversions:", failed_date_conversions)
print("Missing after conversion:", date_parsed.isna().sum())
print("Min date:", date_parsed.min())
print("Max date:", date_parsed.max())

Nonmissing date source values: 109660
Successful date conversions: 109660
Failed date conversions: 0
Missing after conversion: 141875
Min date: 2020-10-19 00:00:00
Max date: 2026-08-11 00:00:00


In [8]:
for col in ["County", "Language", "Annotation Code", "Status"]:
    print(f"\n--- {col} ---")
    print("Unique values:", df_raw[col].nunique())
    print(df_raw[col].value_counts(dropna=False).head(20))


--- County ---
Unique values: 60
County
Los Angeles       6091
Santa Clara       6082
San Diego         6081
Contra Costa      6080
Sacramento        6080
Riverside         6076
Alameda           6075
San Francisco     6065
Orange            6056
San Mateo         6004
San Bernardino    6002
Fresno            5990
Yolo              5981
Solano            5966
Ventura           5900
Sonoma            5788
Placer            5738
Stanislaus        5725
Santa Barbara     5678
San Joaquin       5650
Name: count, dtype: int64

--- Language ---
Unique values: 34
Language
English            11603
Spanish            11601
Missing/Unknown    11572
Other Non-Eng      10848
ASL                 9996
Cantonese           9907
Vietnamese          9751
Tagalog             9750
Mandarin            9593
Russian             9442
Arabic              9196
Korean              9067
Cambodian           8916
Farsi               8746
Portuguese          8386
Thai                8148
Lao                 8088
Oth

In [9]:
print("County special values:")
print(df_raw.loc[df_raw["County"].isin(["STATEWIDE", "Invalid County Code"]), "County"].value_counts())

print("\nLanguage values with possible whitespace:")
print(df_raw.loc[df_raw["Language"].str.startswith(" "), "Language"].value_counts())

print("\nAnnotation Code values:")
print(df_raw["Annotation Code"].value_counts(dropna=False))

print("\nStatus values:")
print(df_raw["Status"].value_counts(dropna=False))

County special values:
County
STATEWIDE              200
Invalid County Code      4
Name: count, dtype: int64

Language values with possible whitespace:
Language
Unknown    2573
Name: count, dtype: int64

Annotation Code values:
Annotation Code
     157211
1     93681
2       643
Name: count, dtype: int64

Status values:
Status
F    235009
P     16526
Name: count, dtype: int64


## Item 5 - Standardize Text

The main text inconsistency identified during inspection is the leading whitespace in the language label ` Unknown`.

Cleaning rule:
- Trim the leading whitespace so that ` Unknown` becomes `Unknown`.
- Preserve the original source value separately.
- Do not merge `Unknown` with `Missing/Unknown` because the source does not establish that they are equivalent.

In [10]:
df_clean = df_raw.copy()

df_clean["Language_original"] = df_clean["Language"]
df_clean["Language"] = df_clean["Language"].str.strip()

changed_language = (
    df_clean["Language_original"] != df_clean["Language"]
)

print("Rows changed:", changed_language.sum())

print("\nBefore:")
print(df_clean.loc[changed_language, "Language_original"].value_counts())

print("\nAfter:")
print(df_clean.loc[changed_language, "Language"].value_counts())

Rows changed: 2573

Before:
Language_original
Unknown    2573
Name: count, dtype: int64

After:
Language
Unknown    2573
Name: count, dtype: int64


In [11]:
key_cols = ["Month of Eligibility", "County", "Language"]

duplicate_keys_after = df_clean.duplicated(
    subset=key_cols,
    keep=False
)

print("Repeated key rows after text standardization:", duplicate_keys_after.sum())

print("\nUnknown count:")
print((df_clean["Language"] == "Unknown").sum())

print("\nMissing/Unknown count:")
print((df_clean["Language"] == "Missing/Unknown").sum())

print("\nUnique language labels after standardization:")
print(df_clean["Language"].nunique())

Repeated key rows after text standardization: 0

Unknown count:
2573

Missing/Unknown count:
11572

Unique language labels after standardization:
34


## Item 6 - Investigate Duplicates

Duplicates are checked in two ways:

1. Exact duplicate rows across all original source fields.
2. Repeated business keys using `Month of Eligibility`, `County`, and cleaned `Language`.

These checks are performed separately because repeated business keys are not necessarily exact duplicate rows.

In [12]:
exact_duplicate_rows = df_raw.duplicated(keep=False)
exact_duplicate_excess = df_raw.duplicated().sum()

key_cols = ["Month of Eligibility", "County", "Language"]

repeated_key_rows = df_clean.duplicated(
    subset=key_cols,
    keep=False
)

repeated_key_excess = df_clean.duplicated(
    subset=key_cols
).sum()

print("Exact duplicate rows involved:", exact_duplicate_rows.sum())
print("Exact duplicate excess rows:", exact_duplicate_excess)

print("\nRepeated business-key rows involved:", repeated_key_rows.sum())
print("Repeated business-key excess rows:", repeated_key_excess)

Exact duplicate rows involved: 0
Exact duplicate excess rows: 0

Repeated business-key rows involved: 0
Repeated business-key excess rows: 0


## Item 7 - Handle Missing Data

Missing values are evaluated using the source representation rather than relying only on pandas null values.

Important distinctions:
- Suppressed `Total Eligibles` values are unavailable source values, not zeros.
- Blank annotation fields indicate that no annotation was recorded.
- `Date of Data` contains both blank values and a dot sentinel, which are treated as unavailable metadata for cleaning.
- `Difference` is entirely empty and its meaning is not inferred.

In [13]:
missing_summary_before = pd.DataFrame({
    "Column": [
        "Total Eligibles",
        "Annotation Code",
        "Annotation Description",
        "Date of Data",
        "Difference"
    ],
    "Missing before": [
        df_raw["Total Eligibles"].str.strip().eq("").sum(),
        df_raw["Annotation Code"].eq("").sum(),
        df_raw["Annotation Description"].eq("").sum(),
        (
            df_raw["Date of Data"].str.strip().isin(["", "."])
        ).sum(),
        df_raw["Difference"].str.strip().eq("").sum()
    ]
})

missing_summary_before

,Column,Missing before
0,Total Eligibles,94324
1,Annotation Code,157211
2,Annotation Description,157011
3,Date of Data,141875
4,Difference,251535


In [14]:
# Normalize missing values without dropping rows

df_clean["Total Eligibles"] = total_numeric.astype("Int64")

df_clean["Annotation Code"] = (
    df_clean["Annotation Code"]
    .replace("", pd.NA)
)

df_clean["Annotation Description"] = (
    df_clean["Annotation Description"]
    .replace("", pd.NA)
)

df_clean["Date of Data"] = date_parsed

df_clean["Difference"] = (
    df_clean["Difference"]
    .str.strip()
    .replace("", pd.NA)
)

missing_summary_after = pd.DataFrame({
    "Column": [
        "Total Eligibles",
        "Annotation Code",
        "Annotation Description",
        "Date of Data",
        "Difference"
    ],
    "Missing after": [
        df_clean["Total Eligibles"].isna().sum(),
        df_clean["Annotation Code"].isna().sum(),
        df_clean["Annotation Description"].isna().sum(),
        df_clean["Date of Data"].isna().sum(),
        df_clean["Difference"].isna().sum()
    ]
})

missing_summary_after

,Column,Missing after
0,Total Eligibles,94324
1,Annotation Code,157211
2,Annotation Description,157011
3,Date of Data,141875
4,Difference,251535


## Item 8 - Check Suspicious Values

Suspicious values are reviewed before deciding whether to correct, keep, or exclude them from specific analyses.

The main checks include:
- non-county geography labels such as `STATEWIDE`
- unresolved geography values such as `Invalid County Code`
- unusually large published eligibility counts
- invalid or impossible dates

In [15]:
print("STATEWIDE rows:")
print((df_clean["County"] == "STATEWIDE").sum())

print("\nInvalid County Code rows:")
print((df_clean["County"] == "Invalid County Code").sum())

print("\nInvalid County Code records:")
display(
    df_clean.loc[
        df_clean["County"] == "Invalid County Code",
        [
            "Month of Eligibility",
            "County",
            "Language",
            "Total Eligibles",
            "Annotation Code",
            "Annotation Description",
            "Status",
            "Date of Data"
        ]
    ]
)

print("\nLargest published Total Eligibles values:")
display(
    df_clean.loc[df_clean["Total Eligibles"].notna()]
    .sort_values("Total Eligibles", ascending=False)
    .head(10)[
        [
            "Month of Eligibility",
            "County",
            "Language",
            "Total Eligibles",
            "Status"
        ]
    ]
)

STATEWIDE rows:
200

Invalid County Code rows:
4

Invalid County Code records:


,Month of Eligibility,County,Language,Total Eligibles,Annotation Code,Annotation Description,Status,Date of Data
15421,2025-09,Invalid County Code,English,<NA>,1,Cell suppressed for small numbers,P,2026-08-11
223773,2011-12,Invalid County Code,English,<NA>,1,Cell suppressed for small numbers,F,NaT
223774,2011-12,Invalid County Code,Spanish,<NA>,1,Cell suppressed for small numbers,F,NaT
237843,2010-12,Invalid County Code,English,<NA>,1,Cell suppressed for small numbers,F,NaT



Largest published Total Eligibles values:


,Month of Eligibility,County,Language,Total Eligibles,Status
53498,2023-05,STATEWIDE,STATEWIDE,16051565,F
54840,2023-04,STATEWIDE,STATEWIDE,15981071,F
56178,2023-03,STATEWIDE,STATEWIDE,15897515,F
57520,2023-02,STATEWIDE,STATEWIDE,15798053,F
58860,2023-01,STATEWIDE,STATEWIDE,15717809,F
52156,2023-06,STATEWIDE,STATEWIDE,15672367,F
49479,2023-08,STATEWIDE,STATEWIDE,15638672,F
50819,2023-07,STATEWIDE,STATEWIDE,15612841,F
60191,2022-12,STATEWIDE,STATEWIDE,15586919,F
61525,2022-11,STATEWIDE,STATEWIDE,15474582,F


In [16]:
invalid_eligibility_dates = month_parsed.isna().sum()

invalid_data_dates = (
    date_clean.notna() & date_parsed.isna()
).sum()

print("Invalid Month of Eligibility values:", invalid_eligibility_dates)
print("Invalid nonmissing Date of Data values:", invalid_data_dates)

Invalid Month of Eligibility values: 0
Invalid nonmissing Date of Data values: 0


## Item 10 - Reconcile Changes

Reconciliation checks confirm whether cleaning changed the number of records or important totals.

No rows are expected to be dropped in this workflow. Text standardization, type conversion, and missing-value normalization should preserve the original row count.

In [17]:
starting_rows = len(df_raw)
final_rows = len(df_clean)

published_total_before = total_numeric.sum(skipna=True)
published_total_after = df_clean["Total Eligibles"].sum(skipna=True)

print("Starting row count:", starting_rows)
print("Final row count:", final_rows)
print("Row difference:", final_rows - starting_rows)

print("\nPublished Total Eligibles sum before cleaning:", published_total_before)
print("Published Total Eligibles sum after cleaning:", published_total_after)
print("Difference in published total:", published_total_after - published_total_before)

Starting row count: 251535
Final row count: 251535
Row difference: 0

Published Total Eligibles sum before cleaning: 4914166969.0
Published Total Eligibles sum after cleaning: 4914166969
Difference in published total: 0.0


## Item 11 - Check Calculations

Published eligibility calculations use only nonmissing `Total Eligibles` values.

Suppressed counts remain missing and are not treated as zero. Therefore:
- sums include only published counts,
- averages use only records with published counts,
- record counts are reported with each metric,
- missing or suppressed values reduce coverage and can cause totals to understate the true population.

The current file does not provide a verified county-level denominator for calculating exact language shares, so exact county language percentages are not calculated from this dataset alone.

In [18]:
published_mask = df_clean["Total Eligibles"].notna()

published_record_count = published_mask.sum()
suppressed_record_count = df_clean["Total Eligibles"].isna().sum()

published_sum = df_clean.loc[published_mask, "Total Eligibles"].sum()
published_average = df_clean.loc[published_mask, "Total Eligibles"].mean()

missing_county_keys = df_clean["County"].str.strip().eq("").sum()
missing_language_keys = df_clean["Language"].str.strip().eq("").sum()
missing_month_keys = df_clean["Month of Eligibility"].str.strip().eq("").sum()

print("Records used for published sum:", published_record_count)
print("Records excluded because count is unavailable:", suppressed_record_count)
print("Published Total Eligibles sum:", published_sum)
print("Published Total Eligibles average:", published_average)

print("\nMissing Month keys:", missing_month_keys)
print("Missing County keys:", missing_county_keys)
print("Missing Language keys:", missing_language_keys)

Records used for published sum: 157211
Records excluded because count is unavailable: 94324
Published Total Eligibles sum: 4914166969
Published Total Eligibles average: 31258.416834699863

Missing Month keys: 0
Missing County keys: 0
Missing Language keys: 0


## Item 12 - Verify Individual Records

Five records were selected to cover both ordinary and difficult cases.

1. **Published numeric count with comma formatting**
   - Expected: a valid published count such as `2,724` should convert to integer `2724` with no change in meaning.

2. **Suppressed small-number record (`Annotation Code = 1`)**
   - Expected: the eligibility count should remain missing and should not be converted to zero.

3. **Suppressed complementary-cell record (`Annotation Code = 2`)**
   - Expected: the eligibility count should remain missing and should not be inferred.

4. **Language value with leading whitespace**
   - Expected: ` Unknown` should become `Unknown`, while remaining separate from `Missing/Unknown`.

5. **Invalid County Code record**
   - Expected: the record should remain in the dataset with `County = Invalid County Code`; no county should be guessed or reassigned.

In [19]:
# Select five records for individual verification

# 1. Published count with comma formatting
idx1 = df_raw[
    (df_raw["Month of Eligibility"] == "2026-08") &
    (df_raw["County"] == "Alameda") &
    (df_raw["Language"] == "Arabic")
].index[0]

# 2. Suppressed small-number record: Annotation Code 1
idx2 = df_raw[
    (df_raw["Annotation Code"] == "1")
].index[0]

# 3. Suppressed complementary-cell record: Annotation Code 2
idx3 = df_raw[
    (df_raw["Annotation Code"] == "2")
].index[0]

# 4. Leading-whitespace language record
idx4 = df_raw[
    (df_raw["Language"] == " Unknown")
].index[0]

# 5. Invalid County Code record
idx5 = df_raw[
    (df_raw["County"] == "Invalid County Code")
].index[0]


record_checks = pd.DataFrame([
    {
        "Check": "Published count with comma",
        "Original": df_raw.loc[idx1, "Total Eligibles"],
        "Expected": 2724,
        "Actual": df_clean.loc[idx1, "Total Eligibles"],
        "Match": df_clean.loc[idx1, "Total Eligibles"] == 2724
    },
    {
        "Check": "Suppressed small number (Code 1)",
        "Original": df_raw.loc[idx2, "Total Eligibles"],
        "Expected": "Missing",
        "Actual": "Missing" if pd.isna(df_clean.loc[idx2, "Total Eligibles"]) else df_clean.loc[idx2, "Total Eligibles"],
        "Match": pd.isna(df_clean.loc[idx2, "Total Eligibles"])
    },
    {
        "Check": "Suppressed complementary cell (Code 2)",
        "Original": df_raw.loc[idx3, "Total Eligibles"],
        "Expected": "Missing",
        "Actual": "Missing" if pd.isna(df_clean.loc[idx3, "Total Eligibles"]) else df_clean.loc[idx3, "Total Eligibles"],
        "Match": pd.isna(df_clean.loc[idx3, "Total Eligibles"])
    },
    {
        "Check": "Leading-whitespace language",
        "Original": repr(df_raw.loc[idx4, "Language"]),
        "Expected": "Unknown",
        "Actual": df_clean.loc[idx4, "Language"],
        "Match": df_clean.loc[idx4, "Language"] == "Unknown"
    },
    {
        "Check": "Invalid County Code retained",
        "Original": df_raw.loc[idx5, "County"],
        "Expected": "Invalid County Code",
        "Actual": df_clean.loc[idx5, "County"],
        "Match": df_clean.loc[idx5, "County"] == "Invalid County Code"
    }
])

record_checks

,Check,Original,Expected,Actual,Match
0,Published count with comma,"2,724",2724,2724,True
1,Suppressed small number (Code 1),,Missing,Missing,True
2,Suppressed complementary cell (Code 2),,Missing,Missing,True
3,Leading-whitespace language,' Unknown',Unknown,Unknown,True
4,Invalid County Code retained,Invalid County Code,Invalid County Code,Invalid County Code,True


## Item 13 - Test Reproducibility

The kernel was restarted and all notebook cells were run from the original raw CSV.

The notebook completed without errors and reproduced the same key results, including:
- 251,535 input and final rows
- 0 exact duplicate rows
- 0 repeated business-key rows
- 2,573 standardized `Unknown` language records
- 94,324 unavailable/suppressed eligibility counts
- published `Total Eligibles` sum of 4,914,166,969
- all five individual record checks matching expected results

## Item 14 - Document Remaining Limitations

Remaining limitations after cleaning:

1. The dataset does not provide a verified county-level total denominator for calculating exact language shares. Published language counts cannot simply be summed because many values are suppressed.

2. There are 94,324 suppressed or unavailable `Total Eligibles` values. These records are retained as missing, so published totals and averages may understate the true population.

3. Four records contain `Invalid County Code`. These records are retained as unresolved and cannot be assigned to a specific county without additional source information.

4. `Unknown` and `Missing/Unknown` remain separate language categories because the source does not establish that they are equivalent.

5. `Date of Data` contains substantial missing metadata, and the exact meaning of the dot sentinel should be confirmed from source documentation.

6. `Difference` is entirely empty in this source file, so its intended meaning and units remain unresolved.

These limitations should be considered when interpreting county rankings, trends, and any future dashboard calculations.

In [20]:
sample_parts = [
    df_clean[df_clean["Total Eligibles"].notna()].head(5),
    df_clean[df_clean["Annotation Code"] == "1"].head(3),
    df_clean[df_clean["Annotation Code"] == "2"].head(3),
    df_clean[df_clean["Language"] == "Unknown"].head(2),
    df_clean[df_clean["Language"] == "Missing/Unknown"].head(2),
    df_clean[df_clean["County"] == "STATEWIDE"].head(2),
    df_clean[df_clean["County"] == "Invalid County Code"].head(3)
]

cleaned_sample = (
    pd.concat(sample_parts)
    .drop_duplicates()
    .head(20)
)

cleaned_sample

,Month of Eligibility,County,Language,Total Eligibles,Annotation Code,Annotation Description,Status,Date of Data,Difference,Language_original
0,2026-08,Alameda,ASL,132,NaN,NaN,P,2026-08-11,NaN,ASL
1,2026-08,Alameda,Arabic,2724,NaN,NaN,P,2026-08-11,NaN,Arabic
2,2026-08,Alameda,Armenian,35,NaN,NaN,P,2026-08-11,NaN,Armenian
3,2026-08,Alameda,Cambodian,715,NaN,NaN,P,2026-08-11,NaN,Cambodian
4,2026-08,Alameda,Cantonese,23765,NaN,NaN,P,2026-08-11,NaN,Cantonese
8,2026-08,Alameda,Hebrew,<NA>,1,Cell suppressed for small numbers,P,2026-08-11,NaN,Hebrew
10,2026-08,Alameda,Hmong,<NA>,1,Cell suppressed for small numbers,P,2026-08-11,NaN,Hmong
12,2026-08,Alameda,Italian,<NA>,1,Cell suppressed for small numbers,P,2026-08-11,NaN,Italian
32,2026-08,Alpine,English,<NA>,2,Cell suppressed for complementary cell,P,2026-08-11,NaN,English
281,2026-08,Inyo,Missing/Unknown,<NA>,2,Cell suppressed for complementary cell,P,2026-08-11,NaN,Missing/Unknown


In [21]:
cleaned_sample.to_csv(
    "cleaned_sample.csv",
    index=False
)

print("Saved cleaned_sample.csv with", len(cleaned_sample), "rows")

Saved cleaned_sample.csv with 20 rows
